# Phase 12: Final Results & Report Generation
**Objective**: Produce the final tables, figures and the written results **only from recorded outputs** of Phases 2-11 (no number is typed in by hand). Every result sentence is generated from the computed statistics by rule (`higher` / `lower` only if the 95% interval of the paired difference excludes 0; otherwise `no significant difference`), so the text cannot claim an improvement the data do not support.

**Outputs** (`results/`): `final_report.md`, `final_thesis_report.json` (all reported numbers + provenance hashes of every source file), `figures/*.png`, `tables/*.csv`. A verification step recomputes the headline numbers from the raw prediction files and checks them against the report.

In [ ]:
# Required Imports
import os
import sys
import json
import glob
import shutil
import hashlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

In [ ]:
# 1. Load every recorded artifact (essential files must exist; optional ones are reported as 'not recorded')
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
config = json.load(open(os.path.join(BASE_DIR, "config.json")))
context_config = json.load(open(os.path.join(BASE_DIR, "context_config.json")))
model_config = json.load(open(os.path.join(BASE_DIR, "model_config.json")))
RESULTS_DIR, DATA_DIR = config["paths"]["results_dir"], config["paths"]["data_dir"]
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)
from meld_eval import describe_diff, paired_stats, holm, mean_std

FIG_DIR, TAB_DIR = os.path.join(RESULTS_DIR, "figures"), os.path.join(RESULTS_DIR, "tables")
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(TAB_DIR, exist_ok=True)
SOURCES = {}           # name -> path (provenance)

def need(fn, kind="json"):
    path = os.path.join(RESULTS_DIR, fn)
    assert os.path.exists(path), f"Missing required artifact results/{fn} -- run the earlier phases first."
    SOURCES[fn] = path
    return json.load(open(path)) if kind == "json" else pd.read_csv(path)

def optional(path, kind="json"):
    if not os.path.exists(path):
        return None
    SOURCES[os.path.relpath(path, BASE_DIR)] = path
    return json.load(open(path)) if kind == "json" else pd.read_csv(path)

p9_cfg = need("phase9_config.json"); p9_sel = need("phase9_selected_lambda.json"); p9_idx = need("phase9_run_index.json")
p9_sum = need("phase9_val_summary.csv", "csv"); p9_diag = need("phase9_weight_diagnostics.csv", "csv")
p10 = need("phase10_test_results.json"); p10_boot = need("phase10_bootstrap.csv", "csv"); p10_runs = need("phase10_test_runs.csv", "csv")
p11 = need("phase11_results.json")
rob_sum = need("phase11/robustness_summary.csv", "csv"); rob_runs = need("phase11/robustness_runs.csv", "csv")
rob_pair = need("phase11/robustness_paired.csv", "csv"); rel_resp = need("phase11/reliability_response.csv", "csv")
rel = need("phase11/reliability_analysis.json"); dist = np.load(os.path.join(RESULTS_DIR, "phase11", "reliability_distributions.npz"))
SOURCES["phase11/reliability_distributions.npz"] = os.path.join(RESULTS_DIR, "phase11", "reliability_distributions.npz")
leak = optional(os.path.join(RESULTS_DIR, "leakage_report.json")); env = optional(os.path.join(RESULTS_DIR, "environment.json"))
dsum = optional(os.path.join(DATA_DIR, "dataset_summary.json")); media = optional(os.path.join(RESULTS_DIR, "missing_media_report.json"))
cls_dist = optional(os.path.join(RESULTS_DIR, "class_distribution.csv"), "csv")

BASELINE, PROPOSED, SEEDS = p10["baseline"], p10["proposed"], p10["seeds"]
EMOTIONS, MODS = config["emotions"], p11["modalities"]
agg, paired = p10["aggregate"], p10["paired_proposed_minus_baseline"]
nS = len(SEEDS)
print(f"Loaded artifacts for baseline='{BASELINE}' vs proposed='{PROPOSED}', {nS} seeds.")

In [ ]:
# 2. Helpers: number formatting and rule-based sentences
def pm(mean, std, nd=4):
    if mean is None or (isinstance(mean, float) and np.isnan(mean)):
        return "n/a"
    return f"{mean:.{nd}f}" if std is None or (isinstance(std, float) and np.isnan(std)) else f"{mean:.{nd}f} ± {std:.{nd}f}"

def A(name, k):
    return pm(agg[name][k]["mean"], agg[name][k]["std"])

def diff_sentence(stats):
    return describe_diff(stats)

def table_md(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(lines)

In [ ]:
# 3. Figures (all drawn from recorded files)
def save_fig(fig, name):
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=200, bbox_inches="tight"); plt.close(fig)
    return path

COL = {BASELINE: "#2b5c8f", PROPOSED: "#d95f02"}
LS = {BASELINE: dict(linestyle="--", marker="s", markersize=7, alpha=0.9), PROPOSED: dict(linestyle="-", marker="o", markersize=4)}
figures = {}

# F1: learning curves (validation macro-F1 and training loss), mean +/- std over seeds
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name in (BASELINE, PROPOSED):
    logs = [[json.loads(l) for l in open(os.path.join(r["run_dir"], "log.jsonl"))] for r in p9_idx if r["name"] == name]
    L = max(len(l) for l in logs)
    for ax, key, ylab in ((axes[0], "val_macro_f1", "validation macro-F1"), (axes[1], "train_loss", "training loss")):
        M = np.full((len(logs), L), np.nan)
        for i, l in enumerate(logs):
            M[i, :len(l)] = [e[key] for e in l]
        mu, sd = np.nanmean(M, 0), np.nanstd(M, 0, ddof=1) if len(logs) > 1 else np.zeros(L)
        x = np.arange(1, L + 1)
        ax.plot(x, mu, color=COL[name], label=name); ax.fill_between(x, mu - sd, mu + sd, color=COL[name], alpha=0.2)
        ax.set_xlabel("epoch"); ax.set_ylabel(ylab); ax.grid(alpha=0.3)
axes[0].legend(); fig.suptitle(f"Learning curves (mean ± std over {nS} seeds)")
figures["learning_curves"] = save_fig(fig, "learning_curves.png")

# F2: lambda sweep on validation (proposed) with the baseline as reference
prop = p9_sum[p9_sum.name.str.startswith("proposed")].copy()
prop["lam"] = prop.name.str.replace("proposed_lam", "").astype(float)
prop = prop.sort_values("lam")
base_row = p9_sum[p9_sum.name == "baseline"].iloc[0]
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.errorbar(prop.lam, prop.best_val_macro_f1_mean, yerr=prop.best_val_macro_f1_std.fillna(0), marker="o", color=COL[PROPOSED], capsize=3, label="proposed")
ax.axhline(base_row.best_val_macro_f1_mean, color=COL[BASELINE], label="baseline")
ax.axhspan(base_row.best_val_macro_f1_mean - np.nan_to_num(base_row.best_val_macro_f1_std), base_row.best_val_macro_f1_mean + np.nan_to_num(base_row.best_val_macro_f1_std), color=COL[BASELINE], alpha=0.15)
ax.set_xscale("log"); ax.set_xlabel("lambda (fixed)"); ax.set_ylabel("validation macro-F1"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Lambda selection (validation only)")
figures["lambda_sweep"] = save_fig(fig, "lambda_sweep.png")

# F3: per-class test F1
fig, ax = plt.subplots(figsize=(9, 4))
x, w = np.arange(len(EMOTIONS)), 0.38
for i, name in enumerate((BASELINE, PROPOSED)):
    ax.bar(x + (i - 0.5) * w, [agg[name][f"f1_{e}"]["mean"] for e in EMOTIONS], w,
           yerr=[np.nan_to_num(agg[name][f"f1_{e}"]["std"]) for e in EMOTIONS], capsize=2, color=COL[name], label=name)
ax.set_xticks(x); ax.set_xticklabels(EMOTIONS, rotation=20); ax.set_ylabel("test F1"); ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_title(f"Per-class test F1 (mean ± std over {nS} seeds)")
figures["per_class_f1"] = save_fig(fig, "per_class_f1.png")

# F4: robustness curves (noise) and missing-modality bars
levels = sorted(rob_sum[rob_sum.kind == "noise"]["level"].unique())
fig, axes = plt.subplots(1, len(MODS), figsize=(4.2 * len(MODS), 3.8), sharey=True)
for ax, m in zip(np.atleast_1d(axes), MODS):
    for name in (BASELINE, PROPOSED):
        clean = rob_sum[(rob_sum.condition == "clean") & (rob_sum.model == name)].iloc[0]
        sub = rob_sum[(rob_sum.kind == "noise") & (rob_sum.modality == m) & (rob_sum.model == name)].sort_values("level")
        xs = [0] + sub.level.tolist(); mu = [clean.macro_f1_mean] + sub.macro_f1_mean.tolist()
        sd = np.nan_to_num(np.array([clean.macro_f1_std] + sub.macro_f1_std.tolist(), dtype=float))
        ax.plot(xs, mu, color=COL[name], label=name, **LS[name]); ax.fill_between(xs, np.array(mu) - sd, np.array(mu) + sd, color=COL[name], alpha=0.2)
    ax.set_title(f"noise on {m}"); ax.set_xlabel("noise level (x train std)"); ax.grid(alpha=0.3)
np.atleast_1d(axes)[0].set_ylabel("test macro-F1"); np.atleast_1d(axes)[0].legend()
figures["robustness_noise"] = save_fig(fig, "robustness_noise.png")

fig, ax = plt.subplots(figsize=(6.5, 4))
conds = ["clean"] + [f"missing_{m}" for m in MODS]
for i, name in enumerate((BASELINE, PROPOSED)):
    sub = rob_sum[(rob_sum.model == name) & rob_sum.condition.isin(conds)].set_index("condition").loc[conds]
    ax.bar(np.arange(len(conds)) + (i - 0.5) * 0.38, sub.macro_f1_mean, 0.38, yerr=np.nan_to_num(sub.macro_f1_std.values.astype(float)), capsize=2, color=COL[name], label=name)
ax.set_xticks(range(len(conds))); ax.set_xticklabels(conds); ax.set_ylabel("test macro-F1"); ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_title("Missing-modality conditions")
figures["robustness_missing"] = save_fig(fig, "robustness_missing.png")

# F5: reliability -- distribution of normalised uncertainty (proposed) and response of the fusion weights
maxlvl = max(levels)
fig, axes = plt.subplots(1, len(MODS), figsize=(4.2 * len(MODS), 3.6))
for i, (ax, m) in enumerate(zip(np.atleast_1d(axes), MODS)):
    series = {"clean": dist[f"u__clean__{PROPOSED}"][:, i], f"noise x{maxlvl:g}": dist[f"u__noise_{m}_{maxlvl:g}__{PROPOSED}"][:, i],
              "missing": dist[f"u__missing_{m}__{PROPOSED}"][:, i]}
    hi = np.percentile(np.concatenate(list(series.values())), 99)
    for lab, v in series.items():
        ax.hist(np.clip(v, 0, hi), bins=40, range=(0, hi), alpha=0.45, density=True, label=lab)
    ax.set_title(f"uncertainty of {m} head"); ax.set_xlabel("normalised u (1 = typical on train)")
np.atleast_1d(axes)[0].legend(); np.atleast_1d(axes)[0].set_ylabel("density")
figures["reliability_distributions"] = save_fig(fig, "reliability_distributions.png")

fig, axes = plt.subplots(1, len(MODS), figsize=(4.2 * len(MODS), 3.6), sharey=False)
for ax, m in zip(np.atleast_1d(axes), MODS):
    for name in (BASELINE, PROPOSED):
        c0 = rob_runs[(rob_runs.condition == "clean") & (rob_runs.model == name)][f"w_{m}"]
        xs, mu, sd = [0.0], [c0.mean()], [c0.std(ddof=1) if len(c0) > 1 else 0.0]
        for lv in levels:
            v = rob_runs[(rob_runs.condition == f"noise_{m}_{lv:g}") & (rob_runs.model == name)][f"w_{m}"]
            xs.append(lv); mu.append(v.mean()); sd.append(v.std(ddof=1) if len(v) > 1 else 0.0)
        ax.errorbar(xs, mu, yerr=sd, color=COL[name], capsize=2, label=name, **LS[name])
    ax.set_title(f"weight on {m} when {m} is noisy"); ax.set_xlabel("noise level (x train std)"); ax.grid(alpha=0.3)
np.atleast_1d(axes)[0].set_ylabel("mean fusion weight"); np.atleast_1d(axes)[0].legend()
figures["weight_response"] = save_fig(fig, "weight_response.png")

# existing figures from earlier phases
for src, dst in ((os.path.join(RESULTS_DIR, "figures", "confusion_matrices.png"), "confusion_matrices"), (os.path.join(RESULTS_DIR, "class_distribution.png"), "class_distribution")):
    if os.path.exists(src):
        figures[dst] = src if os.path.dirname(src) == FIG_DIR else shutil.copy(src, os.path.join(FIG_DIR, os.path.basename(src)))
print({k: os.path.basename(v) for k, v in figures.items()})

In [ ]:
# 4. Result statements generated from the recorded statistics
R = {}   # structured results (written to final_thesis_report.json)

# ---- RQ1: overall performance
boot_pos = int((p10_boot.ci_low > 0).sum()); boot_neg = int((p10_boot.ci_high < 0).sum())
R["RQ1"] = {
    "baseline": {k: agg[BASELINE][k] for k in ("accuracy", "weighted_f1", "macro_f1")},
    "proposed": {k: agg[PROPOSED][k] for k in ("accuracy", "weighted_f1", "macro_f1")},
    "paired": {k: paired[k] for k in ("accuracy", "weighted_f1", "macro_f1")},
    "statement": (f"Macro-F1 (primary metric) on the MELD test set: baseline {A(BASELINE, 'macro_f1')}, proposed {A(PROPOSED, 'macro_f1')}. "
                  f"Paired comparison over seeds (proposed - baseline): macro-F1 {diff_sentence(paired['macro_f1'])}; "
                  f"weighted-F1 {diff_sentence(paired['weighted_f1'])}; accuracy {diff_sentence(paired['accuracy'])}. "
                  f"Per-seed paired dialogue bootstrap of the macro-F1 difference: the 95% interval excludes 0 in the positive direction for {boot_pos} and in the negative direction for {boot_neg} of {nS} seeds."),
}

# ---- RQ2: minority classes
cls = {e: paired[f"f1_{e}"] for e in EMOTIONS}
sig = [(e, s["mean_diff"]) for e, s in cls.items() if not np.isnan(s.get("p_holm_over_classes", np.nan)) and s["p_holm_over_classes"] < 0.05]
R["RQ2"] = {
    "fear": {"baseline": agg[BASELINE]["f1_fear"], "proposed": agg[PROPOSED]["f1_fear"], "paired": paired["f1_fear"]},
    "disgust": {"baseline": agg[BASELINE]["f1_disgust"], "proposed": agg[PROPOSED]["f1_disgust"], "paired": paired["f1_disgust"]},
    "minority_mean": {"baseline": agg[BASELINE]["minority_f1"], "proposed": agg[PROPOSED]["minority_f1"], "paired": paired["minority_f1"]},
    "classes_with_holm_significant_difference": [{"class": e, "mean_diff": d} for e, d in sig],
    "statement": (f"Fear F1: baseline {A(BASELINE, 'f1_fear')}, proposed {A(PROPOSED, 'f1_fear')}; paired difference (proposed - baseline): {diff_sentence(paired['f1_fear'])}. "
                  f"Disgust F1: baseline {A(BASELINE, 'f1_disgust')}, proposed {A(PROPOSED, 'f1_disgust')}; paired difference: {diff_sentence(paired['f1_disgust'])}. "
                  + (f"After Holm correction over the 7 classes, significant per-class differences: "
                     + ", ".join(f"{e} ({d:+.4f})" for e, d in sig) + "." if sig else "After Holm correction over the 7 classes, no per-class F1 difference is significant.")),
}

# ---- RQ3: robustness
paired_deg = rob_pair[(rob_pair.metric == "macro_f1") & (rob_pair.condition != "clean")].copy()
ret = rob_runs[rob_runs.condition != "clean"].copy()
clean_f1 = rob_runs[rob_runs.condition == "clean"].set_index(["model", "seed"])["macro_f1"]
ret["retention"] = [r.macro_f1 / clean_f1[(r.model, r.seed)] for r in ret.itertuples()]
ret_seed = ret.groupby(["model", "seed"])["retention"].mean().unstack(0)
ret_stats = paired_stats(ret_seed[PROPOSED].values, ret_seed[BASELINE].values)
n_cond = len(paired_deg)
better = paired_deg[(paired_deg.p_holm < 0.05) & (paired_deg.mean_diff > 0)]; worse = paired_deg[(paired_deg.p_holm < 0.05) & (paired_deg.mean_diff < 0)]
n_pos = int((paired_deg.mean_diff > 0).sum())
R["RQ3"] = {
    "n_degraded_conditions": n_cond, "noise_levels": p11["noise_levels"],
    "mean_retention": {n: dict(zip(("mean", "std"), mean_std(ret_seed[n].values))) for n in (BASELINE, PROPOSED)},
    "retention_paired": ret_stats,
    "conditions_proposed_significantly_better_holm": better.condition.tolist(), "conditions_proposed_significantly_worse_holm": worse.condition.tolist(),
    "n_conditions_proposed_higher_mean": n_pos,
    "statement": (f"Across {n_cond} degraded test conditions (missing modality, and Gaussian noise at levels {p11['noise_levels']} x train std on each modality), "
                  f"mean macro-F1 retention relative to clean was baseline {pm(*mean_std(ret_seed[BASELINE].values))} and proposed {pm(*mean_std(ret_seed[PROPOSED].values))}; "
                  f"paired difference in retention (proposed - baseline): {diff_sentence(ret_stats)}. "
                  f"The proposed model had a higher mean macro-F1 than the baseline in {n_pos} of {n_cond} conditions; after Holm correction across conditions it was significantly better in {len(better)} "
                  f"({', '.join(better.condition) if len(better) else 'none'}) and significantly worse in {len(worse)} ({', '.join(worse.condition) if len(worse) else 'none'})."),
}

# ---- RQ4: reliability
au = rel["auroc_u_detects_head_errors_clean"]
resp_noise = rel_resp[(rel_resp.kind == "noise") & (rel_resp.level == max(levels))]
resp_miss = rel_resp[rel_resp.kind == "missing"]
neg_dw = rel_resp[(rel_resp.dw_p_holm < 0.05) & (rel_resp.dw_prop_minus_base_mean < 0)]
pos_dw = rel_resp[(rel_resp.dw_p_holm < 0.05) & (rel_resp.dw_prop_minus_base_mean > 0)]
tv = rel["clean_weight_shift_tv"][PROPOSED]
R["RQ4"] = {
    "auroc_u_vs_head_errors_clean": au, "clean_weight_shift_tv": rel["clean_weight_shift_tv"],
    "weight_response_at_max_noise": resp_noise[["condition", f"{BASELINE}_d_w_mean", f"{PROPOSED}_d_w_mean", "dw_prop_minus_base_mean", "dw_p_holm"]].to_dict("records"),
    "weight_response_missing": resp_miss[["condition", f"{BASELINE}_d_w_mean", f"{PROPOSED}_d_w_mean", "dw_prop_minus_base_mean", "dw_p_holm"]].to_dict("records"),
    "conditions_where_proposed_lowers_corrupted_weight_more_holm": neg_dw.condition.tolist(),
    "conditions_where_proposed_lowers_corrupted_weight_less_holm": pos_dw.condition.tolist(),
    "statement": ("AUROC of the normalised uncertainty for detecting each modality head's own errors on clean test data (0.5 = uninformative; proposed model): "
                  + ", ".join(f"{m} {pm(au[PROPOSED][m]['mean'], au[PROPOSED][m]['std'], 3)}" for m in MODS) + ". "
                  f"On clean data the reliability term moves the fusion weights away from the content-gate weights by a mean total-variation distance of {pm(tv['mean'], tv['std'], 4)} (baseline: {pm(rel['clean_weight_shift_tv'][BASELINE]['mean'], rel['clean_weight_shift_tv'][BASELINE]['std'], 4)}). "
                  f"When a modality is corrupted, the proposed model reduced that modality's mean fusion weight more than the baseline did in {len(neg_dw)} of {len(rel_resp)} conditions "
                  f"({', '.join(neg_dw.condition) if len(neg_dw) else 'none'}) and less in {len(pos_dw)} ({', '.join(pos_dw.condition) if len(pos_dw) else 'none'}) after Holm correction."),
}
for k in R:
    print(f"--- {k} ---\n{R[k]['statement']}\n")

In [ ]:
# 5. Tables (CSV + Markdown)
tab = {}
tab["main"] = pd.DataFrame({"Metric": ["Accuracy", "Weighted F1", "Macro F1", "Fear F1", "Disgust F1", "Minority mean F1"],
                            BASELINE: [A(BASELINE, k) for k in ("accuracy", "weighted_f1", "macro_f1", "f1_fear", "f1_disgust", "minority_f1")],
                            PROPOSED: [A(PROPOSED, k) for k in ("accuracy", "weighted_f1", "macro_f1", "f1_fear", "f1_disgust", "minority_f1")],
                            "paired diff (95% CI)": [f"{paired[k]['mean_diff']:+.4f} [{paired[k]['ci_low']:+.4f}, {paired[k]['ci_high']:+.4f}]" if not np.isnan(paired[k]['ci_low']) else f"{paired[k]['mean_diff']:+.4f}"
                                                     for k in ("accuracy", "weighted_f1", "macro_f1", "f1_fear", "f1_disgust", "minority_f1")],
                            "p": [f"{paired[k]['p_value']:.3g}" if not np.isnan(paired[k]['p_value']) else "n/a" for k in ("accuracy", "weighted_f1", "macro_f1", "f1_fear", "f1_disgust", "minority_f1")]})
tab["per_class"] = pd.DataFrame({"class": EMOTIONS, BASELINE: [A(BASELINE, f"f1_{e}") for e in EMOTIONS], PROPOSED: [A(PROPOSED, f"f1_{e}") for e in EMOTIONS],
                                 "mean diff": [f"{paired[f'f1_{e}']['mean_diff']:+.4f}" for e in EMOTIONS],
                                 "p (Holm over classes)": [f"{paired[f'f1_{e}'].get('p_holm_over_classes', np.nan):.3g}" if not np.isnan(paired[f'f1_{e}'].get('p_holm_over_classes', np.nan)) else "n/a" for e in EMOTIONS]})
order = p11["conditions"]
rs = rob_sum.set_index(["condition", "model"])
tab["robustness"] = pd.DataFrame({"condition": order,
                                  BASELINE: [pm(rs.loc[(c, BASELINE), "macro_f1_mean"], rs.loc[(c, BASELINE), "macro_f1_std"]) for c in order],
                                  PROPOSED: [pm(rs.loc[(c, PROPOSED), "macro_f1_mean"], rs.loc[(c, PROPOSED), "macro_f1_std"]) for c in order],
                                  "diff (proposed - baseline)": [f"{rob_pair[(rob_pair.condition == c) & (rob_pair.metric == 'macro_f1')].iloc[0].mean_diff:+.4f}" for c in order],
                                  "p (Holm)": [("n/a" if c == "clean" or np.isnan(rob_pair[(rob_pair.condition == c) & (rob_pair.metric == 'macro_f1')].iloc[0].p_holm) else
                                                f"{rob_pair[(rob_pair.condition == c) & (rob_pair.metric == 'macro_f1')].iloc[0].p_holm:.3g}") for c in order]})
tab["val_selection"] = p9_sum[["name", "n_seeds", "best_val_macro_f1_mean", "best_val_macro_f1_std", "best_val_accuracy_mean", "best_epoch_mean"]].round(4)
tab["auroc"] = pd.DataFrame({n: {m: pm(au[n][m]["mean"], au[n][m]["std"], 3) for m in MODS} for n in (BASELINE, PROPOSED)})
tab["weight_diag"] = p9_diag.groupby("name", sort=False).mean(numeric_only=True).drop(columns="seed").round(5).reset_index()
for k, df in tab.items():
    df.to_csv(os.path.join(TAB_DIR, f"{k}.csv"), index=(k == "auroc"))
print("Saved tables:", list(tab))

In [ ]:
# 6. Assemble the report (Markdown + JSON)
sel = p9_sel
tr = p9_cfg["hp"]
lines = []
add = lines.append
add(f"# Reliability-Aware Adaptive Fusion for Multimodal Conversational Emotion Recognition: Results on MELD\n")
add("*Generated automatically from recorded experiment outputs (Phases 2-11). No number below was entered manually.*\n")

add("## 1. Experimental setup")
tr = p9_cfg["hp"]
add(f"- **Fusion family.** `{p9_cfg.get('family', 'legacy')}`: baseline fusion `{p9_cfg.get('fusion_baseline', 'gated')}`, proposed fusion `{p9_cfg.get('fusion_proposed', 'reliability_gated')}`" + (f"; training augmentation: modality dropout {tr['mod_dropout']}, silent noise {tr['noise_aug_p']} (identical for both models)" if 'mod_dropout' in p9_cfg['hp'] else "") + ".")
add(f"- **Models.** Both models share one architecture (`{BASELINE}`: content-gated fusion; `{PROPOSED}`: reliability-aware gated fusion). They have the same layers, "
    f"dropout and parameter count; the only difference is the term `-lambda * u` in the fusion-gate logits. Per-modality Context Transformers: "
    f"d_model {context_config['d_model']}, {context_config['n_layers']} layers, {context_config['n_heads']} heads, dropout {context_config['dropout']}, speaker mode '{context_config['speaker_mode']}' (positions from dialogue row order).")
_audio_desc = (f"WavLM {config.get('audio_model', '')}, uniform mean of transformer layers 1..L, time-mean pooled, train-fitted z-score"
               if str(config.get('audio_feature_set', '')).startswith('wavlm') else f"openSMILE {config['audio_feature_set']}, train-fitted z-score")
if config.get('audio_ft_feature_set'):
    _audio_desc += '; ' + config['audio_ft_feature_set']
_video_desc = (f"MTCNN largest face + HSEmotion {config.get('video_hsemotion_model', '')}, mean over the {config['video_num_frames']} sampled frames that contain a face; clips without a face are flagged unusable"
               if str(config.get('video_backbone', '')).startswith('mtcnn') else f"DenseNet-121 + GAP, {config['video_num_frames']} frames/clip")
add(f"- **Features.** Text {context_config['in_dims']['text']}-d ({config.get('text_feature_set') or 'RoBERTa'}), audio {context_config['in_dims']['audio']}-d ({_audio_desc}), video {context_config['in_dims']['video']}-d ({_video_desc}).")
add(f"- **Uncertainty.** Per-modality MC-dropout ({model_config['model_kwargs']['mc_passes']} passes, scope '{model_config['model_kwargs']['mc_scope']}'): mean over classes of the variance over passes of the predicted class probabilities; "
    f"normalised by '{model_config['model_kwargs']['u_norm']}' using train-set statistics refreshed each epoch. Fusion weights: softmax(content score - lambda * u); lambda is fixed.")
_lossd = tr.get("loss")
_loss_params = ", ".join(str(k) + "=" + str(v) for k, v in (_lossd or {}).items() if k != "kind")
_loss_desc = ("class-weighted cross-entropy (balanced inverse frequency, train split)" if not _lossd else
              "class-imbalance loss '" + str(_lossd["kind"]) + "' (" + _loss_params + "), chosen on validation data in the loss sweep and identical for both models")
add(f"- **Training.** AdamW (lr {tr['lr']}, weight decay {tr['weight_decay']}), {_loss_desc} + {tr['aux_weight']} x per-modality auxiliary loss, up to {tr['max_epochs']} epochs, "
    f"patience {tr['patience']}, checkpoint and early stopping on **validation macro-F1**. Seeds: {SEEDS} ({nS}). Batch: {p9_cfg['batch_size']} dialogues.")
add(f"- **Lambda selection** (validation, mean macro-F1 over seeds, grid {p9_cfg['lambda_grid']}): selected **{sel['lambda']}** (validation macro-F1 {sel['val_macro_f1_mean']:.4f}).")
if "paired_val_macro_f1_diff_mean" in sel:
    add(f"  Validation paired difference to baseline: {sel['paired_val_macro_f1_diff_mean']:+.4f} (higher in {sel['seeds_where_proposed_higher']} of {sel['n_seeds']} seeds).")
if env:
    add(f"- **Environment.** {env.get('gpu') or env.get('device')}, torch {env.get('torch')}.")
add("")

add("## 2. Data notes")
if dsum:
    add(f"- MELD splits: train {dsum['train_samples']}, dev {dsum['dev_samples']}, test {dsum['test_samples']} utterances ({dsum['total_dialogues']} dialogues).")
if leak is not None:
    n_ov = len(leak["utterance_overlaps"])
    add(f"- Dialogue leakage: no identical dialogue appears in two splits. {n_ov} utterance(s) occur in more than one split and were **kept** in the test set to follow the official protocol "
        f"(details: `results/leakage_report.json`). Macro-F1 without those test rows: "
        + "; ".join(f"{n} {pm(v['mean'], v['std'])}" for n, v in p10["sensitivity_excluding_train_duplicates"]["macro_f1"].items()) + ".")
if media:
    add("- Unusable raw clips (excluded via validity masks, never silently zero-filled): " + ", ".join(f"{s} {v['n_unusable']}/{v['n_utterances']}" for s, v in media.items()) + ".")
ext = {}
for mod_ in ("audio", "video"):
    for sp in ("train", "dev", "test"):
        r_ = optional(os.path.join(RESULTS_DIR, f"{mod_}_extraction_report_{sp}.json"))
        if r_ is not None:
            ext.setdefault(mod_, {})[sp] = ((r_["n_failed"] if "n_failed" in r_ else r_["n"] - r_["n_valid"]), r_["n"])   # video report (faces) has n_valid / n_no_face instead of n_failed
for mod_, d_ in ext.items():
    add(f"- {mod_.capitalize()} feature extraction failures (flagged in validity masks): " + ", ".join(f"{sp} {a}/{b}" for sp, (a, b) in d_.items()) + ".")
if cls_dist is not None:
    add("- Class distribution: see `figures/class_distribution.png`; the minority classes are Fear and Disgust.")
add("")

add("## 3. RQ1 - Overall performance (test)")
add(R["RQ1"]["statement"] + "\n")
add(table_md(tab["main"]) + "\n")
add("![confusion matrices](figures/confusion_matrices.png)\n")
add("## 4. RQ2 - Minority emotion classes")
add(R["RQ2"]["statement"] + "\n")
add(table_md(tab["per_class"]) + "\n")
add("![per-class F1](figures/per_class_f1.png)\n")
add("## 5. RQ3 - Robustness to modality degradation")
add(R["RQ3"]["statement"] + "\n")
add(f"Definitions: *missing* = {p11['missing_definition']}; " + (f"*dead* = {p11['dead_definition']}; " if p11.get("dead_definition") else "") + f"*noise* = {p11['noise_definition']}. Identical corrupted inputs were used for both models and all seeds.\n")
add(table_md(tab["robustness"]) + "\n")
add("![noise robustness](figures/robustness_noise.png)\n\n![missing modality](figures/robustness_missing.png)\n")
add("## 6. RQ4 - Reliability scores and fusion weights")
add(R["RQ4"]["statement"] + "\n")
add("AUROC of normalised uncertainty for detecting modality-head errors (clean test):\n")
add(table_md(tab["auroc"].reset_index().rename(columns={"index": "modality"})) + "\n")
add("![uncertainty distributions](figures/reliability_distributions.png)\n\n![weight response](figures/weight_response.png)\n")
add("Training-time diagnostics (best epochs, mean over seeds; `w_shift_from_content` = total-variation distance between final and content-gate weights):\n")
add(table_md(tab["weight_diag"]) + "\n")
add("## 7. Model selection and training curves")
add(table_md(tab["val_selection"]) + "\n")
add("![lambda sweep](figures/lambda_sweep.png)\n\n![learning curves](figures/learning_curves.png)\n")
add("## 8. Reproducibility")
add(f"All numbers derive from the files listed with SHA-1 hashes in `final_thesis_report.json` (`provenance`). Checkpoints: `checkpoints/phase9*/<model>/seed<k>/best.pt` (the folder name carries the feature-set tag; see `phase9_config.json` -> `run_root`); per-utterance test predictions with uncertainties and fusion weights: `results/phase10/<model>/seed<k>/test_predictions.csv`.")
report_md = "\n".join(lines)
with open(os.path.join(RESULTS_DIR, "final_report.md"), "w") as f:
    f.write(report_md)

def sha1(path):
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return jsonable(o.tolist())
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return None if np.isnan(o) else float(o)
    if isinstance(o, float) and np.isnan(o):
        return None
    return o

final = {
    "thesis_title": "Reliability-Aware Adaptive Fusion for Multimodal Conversational Emotion Recognition: A Study on MELD",
    "baseline": BASELINE, "proposed": PROPOSED, "seeds": SEEDS, "selected_lambda": sel,
    "primary_metrics": {"baseline_macro_f1": agg[BASELINE]["macro_f1"], "proposed_macro_f1": agg[PROPOSED]["macro_f1"]},
    "research_questions": {k: R[k] for k in ("RQ1", "RQ2", "RQ3", "RQ4")},
    "figures": {k: os.path.relpath(v, RESULTS_DIR) for k, v in figures.items()},
    "tables": {k: f"tables/{k}.csv" for k in tab},
    "provenance": {k: {"path": os.path.relpath(p, BASE_DIR), "sha1": sha1(p)} for k, p in SOURCES.items()},
}
with open(os.path.join(RESULTS_DIR, "final_thesis_report.json"), "w") as f:
    json.dump(jsonable(final), f, indent=2)
print(f"Saved final_report.md ({len(report_md.splitlines())} lines) and final_thesis_report.json")

In [ ]:
# 7. Verification: recompute headline numbers from the RAW prediction files and check the report
from sklearn.metrics import f1_score
results = []
def check(name, cond, detail=""):
    detail = str(detail)
    results.append({"test": name, "passed": bool(cond), "detail": detail})
    print(("PASS  " if cond else "FAIL  ") + name + (f"  [{detail}]" if detail else ""))

rep = json.load(open(os.path.join(RESULTS_DIR, "final_thesis_report.json")))
worst = 0.0
for name in (BASELINE, PROPOSED):
    vals = []
    for sd in SEEDS:
        p = pd.read_csv(os.path.join(RESULTS_DIR, "phase10", name, f"seed{sd}", "test_predictions.csv"))
        vals.append(f1_score(p.y_true, p.y_pred, average="macro", labels=list(range(len(EMOTIONS))), zero_division=0))
    key = "baseline" if name == BASELINE else "proposed"
    reported = rep["research_questions"]["RQ1"][key]["macro_f1"]
    worst = max(worst, abs(np.mean(vals) - reported["mean"]), abs((np.std(vals, ddof=1) if len(vals) > 1 else 0) - (reported["std"] or 0)))
check("F1 report macro-F1 (mean, std) equal a recomputation from the raw prediction files", worst < 1e-9, f"max |diff|={worst:.1e}")

fear_ok = True
for name in (BASELINE, PROPOSED):
    v = [f1_score(pd.read_csv(os.path.join(RESULTS_DIR, "phase10", name, f"seed{sd}", "test_predictions.csv")).y_true,
                  pd.read_csv(os.path.join(RESULTS_DIR, "phase10", name, f"seed{sd}", "test_predictions.csv")).y_pred, labels=[EMOTIONS.index("fear")], average=None, zero_division=0)[0] for sd in SEEDS]
    fear_ok &= abs(np.mean(v) - rep["research_questions"]["RQ2"]["fear"]["baseline" if name == BASELINE else "proposed"]["mean"]) < 1e-9
check("F2 reported Fear F1 equals a recomputation from the raw prediction files", fear_ok)

missing = [k for k, v in figures.items() if not (os.path.exists(v) and os.path.getsize(v) > 5000)]
check("F3 every figure exists and is non-trivial", not missing, missing)

import re
md = open(os.path.join(RESULTS_DIR, "final_report.md")).read()
check("F4 report contains all sections", all(h in md for h in ("RQ1", "RQ2", "RQ3", "RQ4", "Reproducibility")))
check("F4b report has no 'nan' / placeholder text", not re.search(r"\bnan\b|TODO|TBD", md, flags=re.I))

stmts = [rep["research_questions"][k]["statement"] for k in ("RQ1", "RQ2", "RQ3", "RQ4")]
check("F5 every research-question statement contains numbers and one of the rule-based verdict phrases (RQ1/RQ2)",
      all(re.search(r"\d", s) for s in stmts) and any(w in stmts[0] for w in ("higher (", "lower (", "no significant difference (", "no inference possible")))
src_ok = all(os.path.exists(os.path.join(BASE_DIR, v["path"])) and len(v["sha1"]) == 40 for v in rep["provenance"].values())
check("F6 provenance lists existing source files with SHA-1 hashes", src_ok and len(rep["provenance"]) >= 10, f"{len(rep['provenance'])} files")

# F7: 'higher'/'lower' verdicts are consistent with the stored interval (rule check)
s = paired["macro_f1"]
expected = "higher" if s["ci_low"] > 0 else ("lower" if s["ci_high"] < 0 else "no significant difference") if not np.isnan(s["ci_low"]) else "no inference possible"
check("F7 RQ1 verdict follows the confidence-interval rule", expected in rep["research_questions"]["RQ1"]["statement"], expected)

failed = [r for r in results if not r["passed"]]
with open(os.path.join(RESULTS_DIR, "report_verification.json"), "w") as f:
    json.dump({"n_tests": len(results), "n_failed": len(failed), "results": results}, f, indent=2)
assert not failed, f"{len(failed)} verification test(s) failed: {[r['test'] for r in failed]}"
print(f"\nAll {len(results)} report checks passed.")
print("Phase 12 Final Results & Summary Reporting completed successfully.")